# Week 7: Action Playbook & Operational Integration (ML-10)

**Track:** Applied Search Intelligence: Google Search Ranking & Discoverability  
**Lane:** Refresh / Content Opportunity Scoring  
**Author:** Altaf (ML Intern)  
**Dataset:** Starter Anonymized Release (`data/raw/content_refresh_anonymized.csv`)  

---

## 1. From Predicted Scores to Human Decisions

A bare probability number ($p = 0.78$) is frustrating to an editor. To make the machine learning model immediately actionable, every scored page is mapped to a discrete **suggested action** backed by diagnostic **reason codes** and an editorial triage checklist.

## 2. The Five Operational Playbook Actions

| Suggested Action | Trigger Criteria | Editorial Triage Protocol | Queue Volume (30k rows) |
|---|---|---|---:|
| **`refresh_and_review_ctr`** | Visible (pos $\le 20$, imp $\ge 500$) + CTR $< 0.5\%$ + decline risk | **SERP Snippet Overhaul:** Rewrite meta title with value hook; optimize meta description; add FAQ schema markup. | **6,657** (22.2%) |
| **`refresh_and_review_engagement`** | Sessions $\ge 30$ + (scroll $< 30\%$ or engagement $< 30\%$) | **Above-the-Fold UX Fix:** Add quick Table of Contents; embed visual diagrams; eliminate mobile popups / layout shifts. | **1,990** (6.6%) |
| **`expand_and_refresh`** | Thin page (<1,200 words) receiving search impressions | **Topical Expansion:** Add missing subtopics; incorporate practical examples; answer secondary search queries. | **82** (0.3%) |
| **`refresh`** | Proven demand (imp $\ge 500$) + decay/staleness signals | **Factual & Link Refresh:** Update stale statistics; replace broken links; update H2/H3 headers; republish timestamp. | **8,178** (27.3%) |
| **`monitor`** | Stable traffic or low demand (<100 impressions) | **Maintain Observation:** No human intervention; re-evaluate in next 30-day reporting window. | **13,093** (43.6%) |

## 3. Confidence Tiering Logic

To protect reviewers from acting on noisy telemetry, we apply a multi-condition confidence filter:

```python
def assign_confidence(row, high_threshold=80.0, medium_threshold=60.0):
    if (
        row["final_refresh_score"] >= high_threshold
        and row["impressions_90d"] >= 500
        and row["sessions_90d"] >= 10
        and row["best_model_probability"] >= 0.50
    ):
        return "high"      # 3,605 items (12.0%)
    if row["final_refresh_score"] >= medium_threshold:
        return "medium"    # 11,395 items (38.0%)
    return "low"           # 15,000 items (50.0%)
```

### Operational Policy
- **Weekly Priority Queue:** Reviewers work strictly on `high` confidence items ranked 1 through 50.
- **Medium Confidence:** Placed in auxiliary review queue for secondary audit cycles.
- **Low Confidence:** Suppressed from human review queues to prevent editorial fatigue.

## 4. Final Blended Opportunity Score Formulation

To balance predictive decay risk with intuitive domain heuristics, the final opportunity score blends the calibrated Random Forest probability (70% weight) with the normalized baseline score (30% weight):

$$\text{Final Refresh Score} = 100 \cdot \Big(0.70 \cdot \hat{p}_{\text{model}} + 0.30 \cdot S_{\text{baseline}}\Big) \in [0, 100]$$

- **Model Probability (70%):** Drives ranking precision by detecting subtle multi-signal decay.
- **Baseline Score (30%):** Guarantees that items with high search volume and extreme staleness are not prematurely discarded.

## 5. Rubric Pass Checklist (ML-10)

- [x] **Discrete actions mapped:** All 5 operational actions defined with concrete trigger logic.
- [x] **Confidence tiering implemented:** Multi-condition confidence filter isolates 3,605 high-confidence targets.
- [x] **Queue volume analyzed:** Action mix distribution quantified across all 30,000 rows.
- [x] **Step-by-step editorial protocols provided:** Action checklists documented for editorial deployment.